> 📄 English: [langgraph_integration.ipynb](langgraph_integration.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-brightdata--langgraph-integration)

# 使用 LangGraph 与 Bright Data 构建智能网页抓取 Agent

## 概述

本综合教程演示了如何使用 LangGraph 的 ReAct 框架，并与 Bright Data 的原生 LangChain 工具集成，来构建一个智能网页抓取 Agent。该方法将推理能力与强大的网页数据提取相结合，创建出一个能够自主搜索、分析并从网络中提取信息的系统。

现代数据采集工作流需要的不仅仅是简单的抓取工具。它们需要智能 Agent，能够理解上下文、就搜索策略做出决策，并适应不同类型的查询。本教程通过构建一个 ReAct（Reasoning and Acting，推理与行动）Agent 来满足这一需求，该 Agent 将大语言模型的决策能力与 Bright Data 提供的稳健数据采集基础设施结合起来。

## 学习目标

完成本教程后，你将理解如何：

- 借助 Bright Data 的 LangChain 集成实现 LangGraph ReAct Agent
- 配置并使用 Bright Data 的 SERP（Search Engine Results Page，搜索引擎结果页）工具进行智能 Web 搜索
- 设计能够自动推理复杂网页抓取任务的 Agent
- 通过适当的错误管理处理实时数据提取
- 为生产环境构建可扩展的网页抓取工作流

## 架构概览

```
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│   User Query    │───▶│   LangGraph      │───▶│   Bright Data   │
│                 │    │   ReAct Agent    │    │   SERP Tool     │
└─────────────────┘    └──────────────────┘    └─────────────────┘
                              │                          │
                              ▼                          ▼
                       ┌─────────────┐            ┌─────────────┐
                       │   Google    │            │   Search    │
                       │   Gemini    │            │   Results   │
                       │    LLM      │            │   (JSON)    │
                       └─────────────┘            └─────────────┘
                              │                          │
                              └────────┬─────────────────┘
                                       ▼
                               ┌───────────────┐
                               │   Structured  │
                               │   Response    │
                               └───────────────┘
```

系统通过一个协调的工作流来运行：ReAct Agent 接收用户查询，使用语言模型确定合适的搜索策略，通过 Bright Data 的基础设施执行搜索，并将结果综合为结构化的回复。


## 快速开始

请按照以下步骤配置你的开发环境：

1. **注册** Bright Data：访问[此链接](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-brightdata--langgraph-integration&click=brightdata&target=https%3A%2F%2Fbrightdata.com%2F%3Fhs_signup%3D1%26utm_source%3Dbrand%26utm_campaign%3Dbrnd-mkt_github_nirdiamant_logo&text=This%20link)（目前每月免费提供 5k 次 unlocker 请求！）。

    _可参考下方链接的截图，获取分步指引：_

    - ![截图：注册页面](assets/Signup.png)
2. 从[账户设置](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-brightdata--langgraph-integration&click=brightdata-setting&target=https%3A%2F%2Fbrightdata.com%2Fcp%2Fsetting&text=your%20account%20settings)中**复制你的 API key**：

    - ![截图：设置页面](assets/Settings.png)
3. **LLM API key** —— 我们将通过其 API 使用 Google 的 Gemini。前往 [Google AI Studio](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-brightdata--langgraph-integration&click=google&target=https%3A%2F%2Faistudio.google.com%2F&text=Google%20AI%20Studio) 注册以获取你的 API key。

4. **安装所需依赖**并配置环境变量。


## 安装与依赖管理

该实现依赖几个关键库，它们分别提供 Agent 功能的不同方面。理解这些依赖有助于后续排查问题以及扩展系统。

### 核心包安装


In [ ]:
# Install required packages
%pip install langchain-brightdata langchain-google-genai langgraph python-dotenv --quiet
%load_ext autoreload
%autoreload 2

### 环境变量配置

妥善管理 API key 对安全性和功能性都至关重要。下面的单元格演示了如何将你的凭据安全地存储在环境变量中。


In [ ]:
# To export your API keys into a .env file, run the following cell (replace with your actual keys):
!echo "BRIGHT_DATA_API_TOKEN=<your-brightdata-api-key>" >> .env
!echo "GOOGLE_API_KEY=<your-google-api-key>" >> .env

### 库导入与环境验证

本节导入所有必要的库，并验证环境已正确配置了所需的 API key。


In [ ]:
# Import all necessary libraries
from langchain_brightdata import BrightDataSERP
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Verify environment setup
print("Environment setup complete!")
print(f"Bright Data API Key loaded: {'Yes' if os.getenv('BRIGHT_DATA_API_TOKEN') else 'No'}")
print(f"Google API Key loaded: {'Yes' if os.getenv('GOOGLE_API_KEY') else 'No'}")

## 核心组件与架构

理解各个组件及其交互方式，对于有效实现和定制网页抓取 Agent 至关重要。

### LangGraph ReAct Agent 框架

ReAct（Reasoning and Acting，推理与行动）范式代表了 Agent 设计的一大进步。传统方法要么只关注推理，要么只关注动作执行。而 ReAct Agent 将两者能力整合起来，使其既能系统地推理问题，又能采取具体行动来收集信息或操控环境。

在网页抓取场景中，这种整合尤为宝贵。Agent 可以分析用户的查询以判断需要哪类信息，制定合适的搜索策略，执行搜索，分析结果，并可能根据所发现的内容进一步优化方法。与静态的、基于规则的方法相比，这创造出了一个更加智能、更具适应性的抓取系统。

### Bright Data SERP 集成

Bright Data 的搜索引擎结果页（SERP）工具通过其全球代理网络提供企业级的 Web 搜索能力。相比直接调用搜索引擎 API，这套基础设施具有多项优势，包括地域多样性、速率限制管理以及一致的数据格式。

该工具支持包括 Google 和 Bing 在内的多个搜索引擎，并可配置地理位置、语言偏好和结果数量等参数。内置的解析功能会将原始搜索结果转换为结构化数据，便于语言模型处理。

### Google Gemini 语言模型

Google 的 Gemini 模型充当 Agent 的推理引擎。其多模态能力以及在推理任务上的出色表现，使其非常适合分析搜索结果并决定后续行动。该模型理解上下文并生成连贯回复的能力，对于将多个来源的信息综合为有用的洞察至关重要。


## 实现：构建网页抓取 Agent

实现过程包括配置每个组件并将它们整合为一个协同工作的系统。我们将一步步构建该 Agent，并解释每个配置决策背后的理由。

### 语言模型初始化

第一步是建立与 Google Gemini API 的连接，并配置模型参数，以便在网页抓取场景中获得最佳性能。


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",  # Fast and capable model
    temperature=0.1  # Low temperature for consistent, focused responses
)

print("Language model initialized successfully!")

该配置使用 `gemini-2.0-flash`，它在速度与能力之间取得了平衡，这对交互式 Agent 应用尤为重要。较低的 temperature（0.1）确保了回复的一致性和专注性，这对生产环境中 Agent 行为的可靠性至关重要。较高的 temperature 可能会给 Agent 的决策过程引入不必要的随机性。


### Bright Data SERP 工具配置

SERP 工具的配置决定了 Agent 将如何与搜索引擎交互，以及会收到什么类型的结果。


In [ ]:
# Initialize the Bright Data SERP tool
serp_tool = BrightDataSERP(
    search_engine="google",  # Use Google as the search engine
    country="us",           # Search from US perspective
    language="en",          # English language results
    results_count=10,       # Get top 10 results
    parse_results=True,     # Automatically parse and structure results
)

print("Bright Data SERP tool configured successfully!")
print(f"   Search Engine: {serp_tool.search_engine}")
print(f"   Country: {serp_tool.country}")
print(f"   Language: {serp_tool.language}")
print(f"   Results Count: {serp_tool.results_count}")

这些配置参数会显著影响搜索结果的类型和质量。选择 Google 作为主要搜索引擎，是因为其索引全面、结果质量高。将国家设为美国、语言设为英语，是为了优化与英语受众相关的内容，不过你可以针对不同市场或语言进行调整。

结果数量设为 10，在覆盖全面性与处理效率之间取得了良好平衡。更多的结果意味着更广的信息面，但也需要更多的处理时间和 API 资源。自动解析功能确保原始 HTML 响应被转换为结构化数据，让语言模型能够轻松理解和处理。


### ReAct Agent 组装

基础实现的最后一步，是将语言模型与搜索工具组合成一个可用的 ReAct Agent。


In [ ]:
# Create the ReAct agent with the LLM and tools
agent = create_react_agent(
    model=llm,           # The language model we initialized
    tools=[serp_tool],   # List of tools available to the agent
    prompt="""You are a web researcher agnet with access to SERP tool, you will HAVE to use the user query, if no specific, country, language, search engine or specific vertical, choose what's best fit users questions"""
)

print("ReAct Web Scraper Agent created successfully!")
print("Agent is ready to search and analyze web content!")

`create_react_agent()` 函数将语言模型与可用工具整合在一起，创建出一个能够自主决定何时以及如何使用搜索能力的 Agent。prompt 为 Agent 提供了关于其角色和决策过程的指引，确保它理解何时使用搜索工具，以及如何根据用户查询调整策略。


## Agent 测试与验证

用各种类型的查询来测试 Agent，有助于验证其功能并展示其推理能力。

### 基础搜索功能测试

该测试演示了 Agent 如何处理一个直接的信息请求，并利用其搜索能力。


In [ ]:
# Simple search query
print("Testing Basic Search...")
print("="*50)

user_query = "What are the latest developments and news in AI technology in the US?"

# Stream the agent's response to see it thinking step by step
for step in agent.stream(
    {"messages": [("human", user_query)]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

流式输出让你可以实时观察 Agent 的推理过程。你会看到 Agent 如何分析查询、决定使用搜索工具、构造合适的搜索词、执行搜索，并将结果综合为连贯的回复。这种透明度对于理解 Agent 行为和排查问题都很有价值。


## Agent 行为分析

理解 Agent 如何处理不同类型的查询，有助于洞察其决策过程，并优化其性能。

### 查询处理流程

Agent 在处理用户查询时遵循一套系统化的方法：

**初步分析阶段：** Agent 审视用户的问题，判断所需信息的类型、是否需要当前或实时数据，以及搜索范围应当多具体或多宽泛。

**工具选择逻辑：** 基于分析结果，Agent 决定是否使用搜索工具。这一决策取决于多种因素，例如查询是否需要超出模型训练数据的当前信息、问题是否涉及需要验证的具体事实性断言，以及信息请求的详细程度。

**搜索策略形成：** 当确定需要搜索时，Agent 会构造合适的搜索词和参数。这包括从用户查询中提取关键概念、确定最优的搜索词组合，以及选择合适的搜索参数。

**结果综合：** 收到搜索结果后，Agent 会分析来自多个来源的信息，将发现综合为连贯的洞察，提供结构化且相关的回答，并在合适时引用来源。

### 高级配置模式

通过配置修改，基础 Agent 可以针对各种专门用例进行扩展和定制。

**多语言研究配置：**
```python
spanish_serp = BrightDataSERP(
    search_engine="google",
    country="es",      # Spain
    language="es",     # Spanish
    results_count=15,
    parse_results=True,
)

spanish_agent = create_react_agent(llm, [spanish_serp])
```

**替代搜索引擎集成：**
```python
bing_serp = BrightDataSERP(
    search_engine="bing",  # Use Bing instead of Google
    country="us",
    language="en",
    results_count=10,
    parse_results=True,
)

bing_agent = create_react_agent(llm, [bing_serp])
```

**大容量研究配置：**
```python
research_serp = BrightDataSERP(
    search_engine="google",
    country="us",
    language="en",
    results_count=20,  # More results for comprehensive research
    parse_results=True,
)

research_agent = create_react_agent(llm, [research_serp])
```


## 高级实现：研究助手

在基础 Agent 之上，我们可以创建更复杂的研究助手，以更强的能力处理复杂的多维度查询。

### 可配置的研究助手工厂

该实现演示了如何为不同的研究场景创建可复用的 Agent 配置。


In [ ]:
def create_research_assistant(search_engine="google", country="us", language="en"):
    """
    Create a specialized research assistant agent
    
    Args:
        search_engine (str): Search engine to use ("google" or "bing")
        country (str): Country code for localized results
        language (str): Language code for results
    
    Returns:
        Agent configured for research tasks
    """
    
    # Initialize components
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.1)
    
    serp_tool = BrightDataSERP(
        search_engine=search_engine,
        country=country,
        language=language,
        results_count=15,  # More results for thorough research
        parse_results=True,
    )
    
    # Create agent with enhanced capabilities
    agent = create_react_agent(llm, [serp_tool])
    
    print(f"Research Assistant created!")
    print(f"   Engine: {search_engine.title()}")
    print(f"   Location: {country.upper()}")
    print(f"   Language: {language.upper()}")
    
    return agent

# Create the research assistant
research_assistant = create_research_assistant()

### 综合研究查询测试

研究助手可以处理复杂的、多部分的查询，这些查询需要全面的信息收集与分析。


In [ ]:
# Comprehensive research query
research_query = """
Please research the renewable energy market trends for 2024-2025. 
I need information about:
1. Market growth predictions
2. Leading companies and their strategies  
3. Recent technological breakthroughs
4. Government policies affecting the sector
"""

print("Starting comprehensive research...")
print("="*60)

for step in research_assistant.stream(
    {"messages": [("human", research_query)]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## 结论与未来方向

本教程演示了如何构建一个智能网页抓取 Agent，它将大语言模型的推理能力与稳健的网页数据采集基础设施结合起来。最终系统能够自主搜索网络、处理实时信息、跨多个主题进行综合研究、通过多步推理处理复杂查询，并从网页数据中提供结构化、可执行的洞察。

### 主要成果

该实现涵盖了现代 Agent 开发的关键方面，包括完整的网页抓取环境搭建、智能 ReAct Agent 的创建、与 Bright Data 强大搜索能力的集成、错误处理与优化策略，以及专用研究助手的开发。

### 扩展方向

这里呈现的基础系统可以朝多个方向扩展，以创建更复杂、更专业的应用。

**工具集成扩展：** 可以集成更多 Bright Data 工具来扩展 Agent 在搜索之外的能力，包括网页抓取、数据解析和内容提取工具。

**用户界面开发：** 可以构建基于 Web 的界面，让非技术用户也能使用该 Agent，并提供查询历史、结果导出和研究项目管理等功能。

**数据持久化系统：** 实现数据存储方案，可用于研究结果归档、随时间进行趋势分析，以及构建机构知识库。

**自动化与调度：** 可以为 Agent 增加调度能力，用于自动化研究任务、监控特定主题以及生成定期报告。

**领域专业化：** 可以为特定行业或用例开发专用 Agent，配备定制化的 prompt、专门的工具和领域相关知识。

### 生产环境考量

在生产环境中部署 Agent 时，还有几项额外考量变得重要，包括速率限制与 API 用量管理、错误处理与恢复机制、结果缓存与优化、安全与访问控制，以及监控与日志系统。

### 应用领域

本教程演示的技术适用于广泛的现实场景，包括竞争情报与市场监控、学术研究与文献发现、内容策展与趋势分析、价格监控与市场研究，以及新闻聚合与分析。

智能推理与稳健网页数据采集的结合，为自动化信息收集与分析创造了强大的可能性。随着这些技术不断发展，潜在应用将持续扩大，使智能网页抓取 Agent 成为各类组织和研究人员越来越有价值的工具。
